### Importar modulos necessários para acessar os codigos

In [8]:
from Treinos.Treinamento_Unet import treinar_modelo
from Testes.Teste_Unet import avaliar_e_visualizar
from Modelos.UNet_baseline import UNetBaseline
from Modelos.UNet_depth_reduced import UNetDepthReduced
from Modelos.UNet_MobileNet_v2 import UNetMobileNetV2
from Modelos.UNet_MobileNet_v3 import UNetMobileNetV3
from Modelos.UNet_SkipConnections import UNetElementWise
from Modelos.UNet_AttentionGates import UNetAttentionGates
from Modelos.UNet_PSA import UNetPSA
from Modelos.UNet_ResNet34 import UNetResNet34
from Modelos.UNet_SegFormer import SegFormerB0
from Modelos.UNet_MobileNetV3_AttentionGates import UNetMobileNetV3AttentionGates
from Modelos.UNet_MobileNetV3_AttentionGates_DPU import UNetMobileNetV3AttentionGatesDPU
from Modelos.UNet_MobileNet_v3_dpu import UNetMobileNetV3_dpu
from Modelos.HyperStarcop_oficial import carregar_hyperstarcop

## Selecionar modelo

#### Descomente na lista MODELS os modelos que vai treinar. Cada item contém classe, nome e starting_point.


In [9]:
def HyperSTARCOP(in_channels=4, out_channels=1):
    return carregar_hyperstarcop(
        "Modelos_treinados/HyperSTARCOP_oficial/final_checkpoint_model.ckpt",
        "cpu",
    )

MODELS = [
    #(UNetBaseline, "UNET", 0.0),
    #(UNetDepthReduced, "UNetDepthReduced", 0.0),
    #(UNetMobileNetV2, "UNetMobileNetV2", 0.0),
    #(UNetMobileNetV3, "UNetMobileNetV3", 0.0),
    #(UNetElementWise, "UNetElementWise", 0.0),
    #(UNetAttentionGates, "UNetAttentionGates", 0.0),
    #(UNetPSA, "UNetPSA", 0.0),
    #(UNetResNet34, "UNetResNet34", 0.0),
    #(SegFormerB0, "SegFormerB0", 0.0),
    #(UNetMobileNetV3AttentionGates, "UNetMobileNetV3AttentionGates", 0.0),
    #(UNetMobileNetV3AttentionGatesDPU, "UnetMobilenetV3AttentionGates_dpu", 0.0),
    #(UNetMobileNetV3_dpu, "Mobile_Net_v3_dpu", 0.0)
    #(UNetMobileNetV3_dpu, "MobileNet_v3_FocalDiceLoss", 0.0)
    (HyperSTARCOP, "HyperSTARCOP", 0.0)
]


Treinar  Mag1c + RGB

In [ ]:
for MODELO, NOME_MODELO, starting_point in MODELS:
    nome_modelo = NOME_MODELO + "_mag1c_rgb"
    treinar_modelo(MODELO, nome_modelo, starting_point, ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"])


Treinar apenas mag1c

In [ ]:
for MODELO, NOME_MODELO, starting_point in MODELS:
    nome_modelo = NOME_MODELO + "_mag1c"
    treinar_modelo(MODELO, nome_modelo, starting_point, ["mag1c"])


Treinar Varon

In [ ]:
for MODELO, NOME_MODELO, starting_point in MODELS:
    nome_modelo = NOME_MODELO + "_varon"
    treinar_modelo(MODELO, nome_modelo, starting_point, ["ratio_wv3_B7_B5_varon21_sum_c_out", "ratio_wv3_B8_B5_varon21_sum_c_out", "ratio_wv3_B7_B6_varon21_sum_c_out"])


Testar mag1c + RGB

In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_BCEloss_mag1c_rgb"
    avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"])


Testar apenas mag1c

In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_mag1c"
    avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c"])


Testar Varon

In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_varon"
    avaliar_e_visualizar(MODELO, nome_modelo, ["ratio_wv3_B7_B5_varon21_sum_c_out", "ratio_wv3_B8_B5_varon21_sum_c_out", "ratio_wv3_B7_B6_varon21_sum_c_out"])


Treinar todas as bandas + mag1c


In [ ]:

for MODELO, NOME_MODELO, starting_point in MODELS:
    nome_modelo = NOME_MODELO + "_all_bands"
    treinar_modelo(MODELO, nome_modelo, starting_point,["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", 
                                                        "TOA_AVIRIS_640nm", "TOA_AVIRIS_2004nm", "TOA_AVIRIS_2109nm",
                                                        "TOA_AVIRIS_2310nm", "TOA_AVIRIS_2350nm", "TOA_AVIRIS_2360nm"])

Testar todas as bandas + mag1c


In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_all_bands"
    avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", 
                                                "TOA_AVIRIS_640nm", "TOA_AVIRIS_2004nm", "TOA_AVIRIS_2109nm",
                                                "TOA_AVIRIS_2310nm", "TOA_AVIRIS_2350nm", "TOA_AVIRIS_2360nm"])

In [10]:
import torch

from thop import profile


def calcular_gmacs(modelo, device, canais_entrada=4, altura=512, largura=512):
    modelo.eval()
    
    # Cria um tensor vazio (dummy) com a resolução que você quer testar
    # Usamos Batch Size = 1 para padronizar o custo por imagem
    tensor_teste = torch.randn(1, canais_entrada, altura, largura).to(device)
    
    # O thop rastreia o forward pass e conta as operações
    macs, params = profile(modelo, inputs=(tensor_teste, ), verbose=False)
    
    # 1 GMAC = 1 bilhão de MACs
    gmacs = macs / 1e9
    
    print(f"\n--- Análise de Custo Computacional ---")
    print(f"Resolução de Entrada : {altura}x{largura} ({canais_entrada} canais)")
    print(f"Parâmetros Totais    : {params / 1e6:.2f} M")
    print(f"Custo Computacional  : {gmacs:.3f} GMACs")
    
    return gmacs

# Defina o dispositivo e a quantidade de canais do seu dataset atual (mag1c + 3 RGB = 4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
canais_entrada = 4

print("=== Relatório de Custos Computacionais (Inferência) ===")

for ModeloClass, nome_modelo, _ in MODELS:
    print(f"\nAnalisando arquitetura: {nome_modelo}")
    
    # 1. Instancia o modelo do zero na GPU
    modelo_instanciado = ModeloClass(in_channels=canais_entrada, out_channels=1).to(device)
    
    # 2. Calcula os GMACs simulando a resolução final de teste (512x512)
    gmacs_512 = calcular_gmacs(
        modelo=modelo_instanciado, 
        device=device, 
        canais_entrada=canais_entrada, 
        altura=512, 
        largura=512
    )

=== Relatório de Custos Computacionais (Inferência) ===

Analisando arquitetura: HyperSTARCOP

--- Análise de Custo Computacional ---
Resolução de Entrada : 512x512 (4 canais)
Parâmetros Totais    : 6.63 M
Custo Computacional  : 13.604 GMACs
